# Create LoadData CSVs with the paths to IC functions for analysis

In this notebook, we create LoadData CSVs that contains paths to each channel per image set and associated illumination correction `npy` files per channel for CellProfiler to process. 

## Import libraries

In [5]:
import argparse
import pathlib
import pandas as pd
import os
import shutil

import sys

sys.path.append("../utils")
import loaddata_utils as ld_utils
from bandicoot_utils import bandicoot_check


## Set paths

In [6]:
argparse = argparse.ArgumentParser(
    description="Create LoadData CSV files to run CellProfiler on the cluster"
)
argparse.add_argument("--HPC", action="store_true", help="Type of compute to run on")

# Parse arguments
args = argparse.parse_args(args=sys.argv[1:] if "ipykernel" not in sys.argv[0] else [])
HPC = args.HPC

print(f"HPC: {HPC}")


HPC: False


In [7]:
# Set the index directory based on whether HPC is used or not
if HPC:
    # Path for index directory to make loaddata csvs though compute cluster (HPC)
    index_directory = pathlib.Path(
        "/scratch/alpine/jtomkinson@xsede.org/PCCMA_atlas_data/"
    ).resolve(strict=True)
else:
    # Find root directory of the project
    root_dir = pathlib.Path().cwd()

    image_base_dir = bandicoot_check(
        pathlib.Path(os.path.expanduser("~/mnt/bandicoot")).resolve(), root_dir
    )
    # Path for index directory  to make loaddata csv locally
    index_directory = pathlib.Path(
        f"{image_base_dir}/PCCMA_data/Atlas_v2_screen/"
    ).resolve(strict=True)

# Set all paths that are common to both HPC and local
config_dir_path = pathlib.Path("../1.whole_image_qc/load_data_config/").resolve(
    strict=True
)
output_csv_dir = pathlib.Path("./loaddata_csvs/").absolute()
output_csv_dir.mkdir(parents=True, exist_ok=True)
illum_directory = pathlib.Path("../2.illumination_correction/illum_directory/").resolve(
    strict=True
)

# Find all 'Images' folders within the directory (one per plate)
images_folders = sorted(index_directory.rglob("Images"))
print(f"Found {len(images_folders)} 'Images' folders in the directory.")


Found 9 'Images' folders in the directory.


## Create LoadData CSVs with illumination functions for all data

In [8]:
# Define the config path
config_path = config_dir_path / "config.yml"

# Iterate over each plate folder and create a LoadData CSV with illumination correction
for images_folder in images_folders:
    # plate folder is the folder that holds the 'Images' folder
    folder = images_folder.parent

    br00_id = folder.name.split("__")[0]
    plate_name = br00_id

    path_to_output_csv = (
        output_csv_dir / f"{plate_name}_loaddata_original.csv"
    ).absolute()

    path_to_output_with_illum_csv = (
        output_csv_dir / f"{plate_name}_loaddata_with_illum.csv"
    ).absolute()

    illum_output_path = (
        (illum_directory / br00_id).absolute().resolve(strict=True)
    )

    ld_utils.create_loaddata_illum_csv(
        index_directory=images_folder,
        config_path=config_path,
        path_to_output=path_to_output_csv,
        illum_directory=illum_output_path,
        plate_id=br00_id,
        illum_output_path=path_to_output_with_illum_csv,
    )

    print(f"Created LoadData CSV with illumination for {plate_name}")


BR00150701_loaddata_with_illum.csv is created!
The BR00150701_loaddata_original.csv CSV file has been removed as it does not contain the IC functions.
Created LoadData CSV with illumination for BR00150701
BR00150703_loaddata_with_illum.csv is created!
The BR00150703_loaddata_original.csv CSV file has been removed as it does not contain the IC functions.
Created LoadData CSV with illumination for BR00150703
BR00150699_loaddata_with_illum.csv is created!
The BR00150699_loaddata_original.csv CSV file has been removed as it does not contain the IC functions.
Created LoadData CSV with illumination for BR00150699
BR00150702_loaddata_with_illum.csv is created!
The BR00150702_loaddata_original.csv CSV file has been removed as it does not contain the IC functions.
Created LoadData CSV with illumination for BR00150702
BR00150695_loaddata_with_illum.csv is created!
The BR00150695_loaddata_original.csv CSV file has been removed as it does not contain the IC functions.
Created LoadData CSV with ill

In [9]:
# Folder containing the loaddata CSVs
loaddata_dir = output_csv_dir

# Folder containing the platemaps
platemap_dir = pathlib.Path("../0.download_data/metadata/platemaps/").resolve(
    strict=True
)

# Output folder (cleared first so CSVs from a previous run/machine or renamed cell lines are not left behind)
cell_line_output_dir = output_csv_dir / "cell_line_loaddata_csvs"
if cell_line_output_dir.exists():
    shutil.rmtree(cell_line_output_dir)
cell_line_output_dir.mkdir()

# Iterate through loaddata csvs
for loaddata_csv in loaddata_dir.glob("*_loaddata_with_illum.csv"):

    print(f"\nProcessing loaddata: {loaddata_csv.name}")

    plate_id = loaddata_csv.name.split("_")[0]

    # Find matching platemap
    platemap_path = next(platemap_dir.glob(f"*{plate_id}*.csv"))
    print(f"Using platemap: {platemap_path.name}")

    loaddata_df = pd.read_csv(loaddata_csv)
    platemap_df = pd.read_csv(platemap_path)

    # Iterate through each cell line
    for cell_line in platemap_df["cell_line"].unique():

        cell_line_clean = cell_line.replace(" ", "_")

        # Create folder for this cell line
        cell_line_folder = cell_line_output_dir / cell_line_clean
        cell_line_folder.mkdir(exist_ok=True)

        cell_line_platemap = platemap_df[platemap_df["cell_line"] == cell_line][
            ["well_position"]
        ]

        filtered_df = loaddata_df.merge(
            cell_line_platemap,
            left_on="Metadata_Well",
            right_on="well_position",
            how="inner",
        )

        output_file = (
            cell_line_folder / f"{plate_id}_{cell_line_clean}_loaddata_with_illum.csv"
        )

        filtered_df.to_csv(output_file, index=False)

        print(f"Saved: {output_file}")



Processing loaddata: BR00150699_loaddata_with_illum.csv
Using platemap: BR00150699_platemap.csv
Saved: /media/18tbdrive/1.Github_Repositories/pccma_atlas_image_profiling/3.cp_analysis/loaddata_csvs/cell_line_loaddata_csvs/A673/BR00150699_A673_loaddata_with_illum.csv
Saved: /media/18tbdrive/1.Github_Repositories/pccma_atlas_image_profiling/3.cp_analysis/loaddata_csvs/cell_line_loaddata_csvs/CF1500/BR00150699_CF1500_loaddata_with_illum.csv
Saved: /media/18tbdrive/1.Github_Repositories/pccma_atlas_image_profiling/3.cp_analysis/loaddata_csvs/cell_line_loaddata_csvs/CHLA200/BR00150699_CHLA200_loaddata_with_illum.csv
Saved: /media/18tbdrive/1.Github_Repositories/pccma_atlas_image_profiling/3.cp_analysis/loaddata_csvs/cell_line_loaddata_csvs/CHP212/BR00150699_CHP212_loaddata_with_illum.csv
Saved: /media/18tbdrive/1.Github_Repositories/pccma_atlas_image_profiling/3.cp_analysis/loaddata_csvs/cell_line_loaddata_csvs/CHP212-1020/BR00150699_CHP212-1020_loaddata_with_illum.csv
Saved: /media/18tbdr